In [1]:
import os
os.environ["CLEARML_NO_AUTO_CONNECT"] = "1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, BitsAndBytesConfig
from datasets import load_dataset
from peft import UIOrthoLoRAConfig, get_peft_model, PeftModel, VeraConfig
import numpy as np
import torch
import evaluate

In [3]:
# torch.set_printoptions(threshold=torch.inf)  # Display all elements
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# torch.cuda.device_count()

In [4]:
BASE_ID = "roberta-base"
tok  = AutoTokenizer.from_pretrained(BASE_ID, use_fast=True)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_ID,
    num_labels=2,
    device_map="auto")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
uiortholora_cfg = UIOrthoLoRAConfig(
        target_modules=["query", "value"],
        uiortholora_alpha=1.0,
        uiortholora_dropout=0.0,
        num_svalues_to_adapt=10,
        num_svectors_to_adapt=10,
        fan_in_fan_out=False)
model = get_peft_model(base_model, uiortholora_cfg)

num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_adapt:  10
num_svectors_to_adapt:  10
kwargs:  {'uiortholora_alpha': 1.0, 'init_uiortholora_weights': True}
num_svalues_to_a

In [6]:
# vera_cfg = VeraConfig(
#         r=128,
#         target_modules=["query", "value"],
#         fan_in_fan_out=False,)


# model = get_peft_model(base_model, vera_cfg)

In [7]:
# stop

In [8]:
model.classifier.requires_grad_(True)

RobertaClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=2, bias=True)
)

In [9]:
for name, param in model.named_parameters():
    if "classifier" in name:
        print(name)
        print(param.shape)
        print(param.requires_grad)

base_model.model.classifier.dense.weight
torch.Size([768, 768])
True
base_model.model.classifier.dense.bias
torch.Size([768])
True
base_model.model.classifier.out_proj.weight
torch.Size([2, 768])
True
base_model.model.classifier.out_proj.bias
torch.Size([2])
True


In [11]:
for name, module in model.named_modules():
    if hasattr(module, "uiortholora_sigma"):
        print(f"\nAdapter module: {name}")
        for adapter_name in module.uiortholora_sigma.keys():
            print(f"  Adapter name: {adapter_name}")
            print(f"    Σ shape: {module.uiortholora_sigma[adapter_name].shape}")
            print(f"    D shape: {module.uiortholora_D[adapter_name].shape}")
            print(f"    E shape: {module.uiortholora_E[adapter_name].shape}")
            print(f"    U shape: {getattr(module, f'{adapter_name}_U').shape}")
            print(f"    V shape: {getattr(module, f'{adapter_name}_V').shape}")
            print(f"    left unitary shape: {module.uiortholora_left_unitary[adapter_name].weight.shape}")
            print(f"    right unitary shape: {module.uiortholora_right_unitary[adapter_name].weight.shape}")


Adapter module: base_model.model.roberta.encoder.layer.0.attention.self.query
  Adapter name: default
    Σ shape: torch.Size([10])
    D shape: torch.Size([768])
    E shape: torch.Size([768])
    U shape: torch.Size([768, 10])
    V shape: torch.Size([10, 768])
    left unitary shape: torch.Size([10, 10])
    right unitary shape: torch.Size([10, 10])

Adapter module: base_model.model.roberta.encoder.layer.0.attention.self.value
  Adapter name: default
    Σ shape: torch.Size([10])
    D shape: torch.Size([768])
    E shape: torch.Size([768])
    U shape: torch.Size([768, 10])
    V shape: torch.Size([10, 768])
    left unitary shape: torch.Size([10, 10])
    right unitary shape: torch.Size([10, 10])

Adapter module: base_model.model.roberta.encoder.layer.1.attention.self.query
  Adapter name: default
    Σ shape: torch.Size([10])
    D shape: torch.Size([768])
    E shape: torch.Size([768])
    U shape: torch.Size([768, 10])
    V shape: torch.Size([10, 768])
    left unitary shape:

In [12]:
for name, param in model.named_parameters(recurse=True):
    # if param.requires_grad:
    print(name, param.shape, param.requires_grad)

base_model.model.roberta.embeddings.word_embeddings.weight torch.Size([50265, 768]) False
base_model.model.roberta.embeddings.position_embeddings.weight torch.Size([514, 768]) False
base_model.model.roberta.embeddings.token_type_embeddings.weight torch.Size([1, 768]) False
base_model.model.roberta.embeddings.LayerNorm.weight torch.Size([768]) False
base_model.model.roberta.embeddings.LayerNorm.bias torch.Size([768]) False
base_model.model.roberta.encoder.layer.0.attention.self.query.base_layer.weight torch.Size([768, 768]) False
base_model.model.roberta.encoder.layer.0.attention.self.query.base_layer.bias torch.Size([768]) False
base_model.model.roberta.encoder.layer.0.attention.self.query.uiortholora_sigma.default torch.Size([10]) True
base_model.model.roberta.encoder.layer.0.attention.self.query.uiortholora_D.default torch.Size([768]) True
base_model.model.roberta.encoder.layer.0.attention.self.query.uiortholora_E.default torch.Size([768]) True
base_model.model.roberta.encoder.layer.

In [11]:
# def calculate_effective_rank(matrix):
#     s = torch.linalg.svdvals(matrix)
#     p = s / s.sum()
#     effective_rank = int(torch.exp(-torch.sum(p * torch.log(p + 1e-12))))
#     num_small_sv = (s < 1).sum().item()
#     return effective_rank, num_small_sv

# for name, module in model.named_modules():
#     if name.endswith("attention.self.query") or name.endswith("attention.self.value"):
#         base = getattr(module, "base_layer", module)
#         weight = base.weight.detach().cpu()  # Move to CPU to avoid CUDA issues
#         rank = torch.linalg.matrix_rank(weight)
#         effective_rank, num_small_sv = calculate_effective_rank(weight)
#         print(f"{name}: shape={tuple(weight.shape)}, rank={rank.item()}, "
#               f"effective_rank={effective_rank}, sv<1={num_small_sv}; "
#               f"rank to adapt={rank.item() - effective_rank}")


In [12]:
# stop

In [13]:
# model.config.pad_token_id = tok.pad_token_id

# # ---------- data ----------
# raw_ds = load_dataset("glue", "cola")

# def tok_f(ex):
#     return tok(
#         ex["sentence"],
#         truncation=True,
#         padding="max_length",
#         max_length=128
#     )

# cola_ds = load_dataset("glue", "cola")
# cola_ds = cola_ds.map(tok_f, batched=True)
# cola_ds = cola_ds.rename_column("label", "labels")
# cola_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [14]:
def prepare_dataset(tokenizer, max_len=128, task="sst2"):
    ds = load_dataset("glue", task)
    
    def tokenize_function(examples):
        if task in ["sst2", "cola"]:
            # Single sentence tasks
            return tokenizer(
                examples["sentence"],
                truncation=True,
                padding="max_length",
                max_length=max_len
            )
        elif task in ["mrpc", "qnli", "rte", "wnli", "mnli", "qqp", "sts-b"]:
            # Two sentence tasks
            return tokenizer(
                examples["sentence1"],
                examples["sentence2"],
                truncation=True,
                padding="max_length",
                max_length=max_len
            )
    
    ds = ds.map(tokenize_function, batched=True)
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [ ]:
tokenized_dataset = prepare_dataset(tok)

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [16]:
# ---------- trainer ----------
args = TrainingArguments(
        output_dir="uilinlora-cola",
        per_device_train_batch_size=32,
        num_train_epochs=1,
        learning_rate=3e-3,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50)


trainer = Trainer(model=model,
                  args=args,
                  train_dataset=tokenized_dataset["train"],
                  eval_dataset=tokenized_dataset["validation"])

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

ClearML Task: overwriting (reusing) task id=fd9a3ef15a814863a49cfed10bf5ec64
2025-05-20 11:53:45,175 - clearml.Repository Detection - WARNING - Could not read Jupyter Notebook: No module named 'nbconvert'
2025-05-20 11:53:45,175 - clearml.Repository Detection - WARNING - Please install nbconvert using "pip install nbconvert"
2025-05-20 11:53:45,233 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/66c853484d054dd9ad88ce3f12fba2ca/experiments/fd9a3ef15a814863a49cfed10bf5ec64/output/log
2025-05-20 11:53:48,308 - clearml.Task - WARNING - Parameters must be of builtin type (Transformers/accelerator_config[AcceleratorConfig])


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


In [ ]:
stop

In [27]:
# trainer.train()

In [28]:
# stop

In [29]:
# predictions = trainer.predict(tokenized_datasets["validation"])
# logits = predictions.predictions[1]
# preds = np.argmax(logits, axis=-1)
# metric = evaluate.load("accuracy")

# tokenized_datasets["validation"]["labels"]
# metric.compute(predictions=preds, references=tokenized_datasets["validation"]["labels"])

In [30]:
# 

In [31]:
# # after trainer.train()
# adapter_dir = "uilinlora_adapter"
# model.save_pretrained(adapter_dir, safe_serialization=True)  # adapter only
# tok.save_pretrained(adapter_dir)                             # optional, for easy reload

In [32]:
# 

In [33]:
# 

In [34]:
# BASE_ID = "roberta-base"
# base = AutoModelForSequenceClassification.from_pretrained(
#            BASE_ID, num_labels=2, device_map="auto")

# model = PeftModel.from_pretrained(base, "uilinlora_adapter").to(device)
# tokenizer = AutoTokenizer.from_pretrained("uilinlora_adapter", use_fast=True)

In [35]:
# predictions = trainer.predict(tokenized_datasets["validation"])
# logits = predictions.predictions[1]
# preds = np.argmax(logits, axis=-1)
# metric = evaluate.load("accuracy")

# tokenized_datasets["validation"]["labels"]
# metric.compute(predictions=preds, references=tokenized_datasets["validation"]["labels"])

In [36]:
# 

In [37]:
# 

In [38]:
# 

In [39]:
# 

In [40]:
# # # Debugging things

# # core_model   = model.get_base_model()        # → LlamaForSequenceClassification
# # llama_blocks = core_model.model.layers       # → ModuleList of decoder layers
# # qproj_0      = llama_blocks[0].self_attn.q_proj

# # print(type(qproj_0))          # should be your Linear4bit / Linear8bitLt
# # print(qproj_0.weight.shape)   # should be (out, in)  e.g.  (4096, 2048)

# model.train()
# with torch.amp.autocast("cuda"):
#     batch = tok(["hello"], return_tensors="pt").to(0)
#     out = model(**batch, labels=torch.tensor([1]).to(0))

# loss = out.loss.to(torch.float32)
# loss.backward()


# # Check grads manually
# for name, param in model.named_parameters():
#     if param.requires_grad and param.grad is not None:
#         print(f"{name} has non-zero grad: {param.grad.abs().mean().item():.6f}")

# model.train()
# batch = tok(["hello"], return_tensors="pt").to(0)
# out = model(**batch, labels=torch.tensor([1]).to(0))

# print("loss requires grad?", out.loss.requires_grad)
# print("loss grad_fn?", out.loss.grad_fn)

# for n, p in model.named_parameters():
#     if p.requires_grad:
#         print(n, p.shape)

#         # model.train()
# # batch = tok(["hello"], return_tensors="pt").to(0)
# # out = model(**batch, labels=torch.tensor([1]).to(0))
# # print(out.loss.grad_fn)  # should NOT be None



In [41]:
# For non trained model accuracy 0.4919
# For r=128 one epoch lr 3e-3 accuracy 0.932
# For r=128 two epochs lr 3e-3 accuracy 0.939

In [42]:
# 

In [43]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch, evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, BitsAndBytesConfig, Trainer
)
from peft import UILinLoRAConfig, get_peft_model, LoraConfig, TaskType

In [44]:
torch.set_printoptions(threshold=float("inf"))
matthews = evaluate.load("matthews_correlation")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    result = matthews.compute(predictions=preds, references=labels)
    print("MCC result:", result)
    return result

# accuracy = evaluate.load("accuracy")

In [45]:
# ---------------------------  custom trainer  --------------------------- #
class UILinLoRATrainer(Trainer):
    def __init__(self, *args, head_lr=1e-3, adapter_lr=4e-3, **kw):
        super().__init__(*args, **kw)
        self.head_lr, self.adapter_lr = head_lr, adapter_lr

    def create_optimizer(self):                       # two learning rates
        if self.optimizer is None:
            head, adapter = [], []
            for n, p in self.model.named_parameters():
                if p.requires_grad:
                    (head if "classifier" in n else adapter).append(p)
            groups = [{"params": head,    "lr": self.head_lr},
                      {"params": adapter, "lr": self.adapter_lr}]
            self.optimizer = torch.optim.AdamW(groups)
        return self.optimizer

# ---------------------------  helpers  --------------------------- #
# def prepare_sst2_dataset(tokenizer, max_len=128):
#     ds = load_dataset("glue", "sst2")
#     ds = ds.map(
#         lambda ex: tokenizer(ex["sentence"],
#                              truncation=True,
#                              padding="max_length",
#                              max_length=max_len),
#         batched=True)
#     ds = ds.rename_column("label", "labels")
#     ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
#     return ds

def prepare_cola_dataset(tokenizer, max_len=128):
    ds = load_dataset("glue", "cola")
    ds = ds.map(
        lambda ex: tokenizer(
            ex["sentence"],
            truncation=True,
            padding="max_length",
            max_length=max_len,
        ),
        batched=True,
    )
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [ ]:
base_id = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(base_id, use_fast=True)

base = AutoModelForSequenceClassification.from_pretrained(
    base_id, num_labels=2, device_map="auto"
)

uilinlora_cfg = UILinLoRAConfig(
        target_modules=["query", "value"],
        rank=128,
        uilinlora_alpha=1.0,
        uilinlora_dropout=0.0,
        fan_in_fan_out=False,
        init_uilinlora_weights=True,
        task_type=TaskType.SEQ_CLS)

model = get_peft_model(base, uilinlora_cfg)
model.classifier.requires_grad_(True)   # make head trainable
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
data = prepare_cola_dataset(tokenizer)

In [ ]:
train_args = TrainingArguments(
    output_dir="uilinlora-roberta-base-cola",
    per_device_train_batch_size=64,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_matthews_correlation",
    greater_is_better=True,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    logging_steps=50,
    save_total_limit=1,
    seed=42,
)

trainer = UILinLoRATrainer(
    model=model,
    args=train_args,
    # train_dataset=data["train"].select(range(1000)),
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    head_lr=1e-3,
    adapter_lr=4e-3,
)

trainer.train()
# print("Best-epoch accuracy:",
#         trainer.evaluate()["eval_accuracy"])

In [ ]:
stop

In [ ]:
# data["validation"]

In [ ]:
# print(trainer.evaluate().keys())


In [ ]:
# metrics = trainer.evaluate()
# print(metrics)


In [ ]:
# output = trainer.predict(data["validation"])
# print("Predictions:", output)
